## 多轮 Tool Calling + 错误处理 + 带记忆的 Agent
目标：​ 今天结束时，你的 Agent 能连续调用多个工具、优雅处理错误、并且记住对话上下文。

In [23]:
## 从.env 读取 api_key
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

# 如果 api_key 有值（非空、非 None），就打印 api_key加载成功；否则打印 错误:未找到 API Key
print("api_key加载成功" if api_key else "错误：未找到 API Key")

api_key加载成功


## 第 1 步：升级 Agent——支持多轮工具调用

In [4]:
# 昨日的工具函数直接拿来复用
def get_bom_change(bom_id):
    bom_data = {
        "BOM-1001": "影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2",
        "BOM-1002": "影响物料：电池 5000mAh, 外壳 B 款；涉及产线：组装-3",
    }
    return bom_data.get(bom_id, "未找到该 BOM 的变更记录")

def get_material_stock(material_name):
    stock_data = {
        "螺丝 M3": "库存充足，现有 5000 个",
        "钣金件 A12": "库存紧张，仅剩 120 个，建议补货",
        "电池 5000mAh": "库存充足，现有 2000 个",
        "外壳 B 款": "无库存，已停产",
    }
    return stock_data.get(material_name, f"未找到物料 {material_name} 的库存信息")

# 工具定义
tools_v2 = [
    {
        "type": "function",
        "function": {
            "name": "get_bom_change",
            "description": "根据 BOM 编号查询变更影响到的物料和产线",
            "parameters": {
                "type": "object",
                "properties": {
                    "bom_id": {
                        "type": "string",
                        "description": "BOM 编号，如 BOM-1001"
                    }
                },
                "required": ["bom_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_material_stock",
            "description": "查询某个物料的当前库存情况",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_name": {
                        "type": "string",
                        "description": "物料名称，如 螺丝 M3"
                    }
                },
                "required": ["material_name"]
            }
        }
    }
]

available_functions_v2 = {
    "get_bom_change": get_bom_change,
    "get_material_stock": get_material_stock
}

### resp.json所得结果结构如下：（整理昨日所得）
resp.json()  
<mark style="background-color: #D4EDDA;">resp.json()["choices"]</mark>  
<mark style="background-color: #FFF3CD;">resp.json()["choices"][0]["message"]</mark> // 模型的消息，需加入对话历史  
<font color='blue'>resp.json()["choices"][0]["message"]["content"]</font> // 模型所给回答，content 是 null，因为模型要调工具  
<mark style="background-color: #D1ECF1;">resp.json()["choices"][0]["message"]["tool_calls"]</mark> // 这里就是工具调用请求  
<font color='violet'>resp.json()["choices"][0]["message"]["tool_calls"]["id"]</font> // 调用工具后作为消息的一部分返回  
<font color='violet'>resp.json()["choices"][0]["message"]["tool_calls"]["function"]</font>   
resp.json()["choices"][0]["message"]["tool_calls"]["function"]["name"] // 模型决定调用函数  
resp.json()["choices"][0]["message"]["tool_calls"]["function"]["arguments"] // 模型决定调用函数的参数  

{
  'id': 'gen-1791616741-zjTKTUC33dGHcpPIQmUC',
  'object': 'chat.completion',
  'created': 1791616741,
  'model': 'nvidia/nemotron-3.5-lightning:free',
  'provider': 'Nvidia',
  'system_fingerprint': None,
  'service_tier': None,
 <font color='green'> 'choices'</font>:<mark style="background-color: #D4EDDA;"> [{
    'index': 0,
    'logprobs': None,
    'finish_reason': 'tool_calls',
    'native_finish_reason': 'tool_calls',
   <font color='red'> 'message'</font>: <mark style="background-color: #FFF3CD;">{
      'role': 'assistant',
      <font color='blue'>'content'</font>: None,
      'refusal': None,
      'reasoning': 'We need to query get_bom_change with bom_id "BOM-1001". Use function.',
      <font color='blue'>'tool_calls'</font>:<mark style="background-color: #D1ECF1;"> [{
        'type': 'function',
        'index': 0,
        <font color='violet'>'id'</font>: 'call-7d48e76b-664b-4669-8b0c-5863d323e15b',
        <font color='violet'>'function'</font>: {
          'name': 'get_bom_change',
          'arguments': '{"bom_id":"BOM-1001"}'
        }
      }]</mark>,
      'reasoning_details': [{
        'type': 'reasoning.text',
        'text': 'We need to query get_bom_change with bom_id "BOM-1001". Use function.',
        'format': 'unknown',
        'index': 0
      }]
    }</mark>
  }]</mark>,
  'usage': {
    'prompt_tokens': 343,
    'completion_tokens': 57,
    'total_tokens': 400,
    'cost': 0,
    'is_byok': False,
    'prompt_tokens_details': {
      'cached_tokens': 0,
      'cache_write_tokens': 0,
      'audio_tokens': 0,
      'video_tokens': 0
    },
    'cost_details': {
      'upstream_inference_cost': 0,
      'upstream_inference_prompt_cost': 0,
      'upstream_inference_completions_cost': 0
    },
    'completion_tokens_details': {
      'reasoning_tokens': 18,
      'image_tokens': 0,
      'audio_tokens': 0
    }
  }
}

json.loads()‌：JSON 字符串 -> Python 字典/列表‌（读取、解析）  
json.dumps()‌：Python 字典/列表 -> JSON 字符串‌（输出、序列化）

In [6]:
import requests
import json

#  -- 配置 --
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-type": "application/json",
}

def run_agent_multi_turn(user_input, tools, available_functions, max_turns=5):
    """
    支持多轮工具调用的 Agent
    
    参数：
        user_input: string, 用户输入
        tools: list, 工具定义列表
        available_functions: dict, 函数名到实际函数的映射
        max_turns: int, 最大工具调用轮数（防止无限循环）
    
    返回：
        string, 最终回答
    """
    
    messages = [
            {"role":"system", "content":"你是制造企业的 AI 助理，回答简洁专业，用中文。"},
            {"role":"user", "content": user_input}
        ]
    
    turn_count = 0
    
    while turn_count < max_turns:
        turn_count += 1
        print(f"\n--- 第 {turn_count} 轮调用 ---")
    
        # 调模型
        data = json.dumps({
            "model": "nvidia/nemotron-3.5-lightning:free",
            "messages":messages,
            "tools":tools,
            "tool_choice":"auto"
        })

        resp = requests.post(url=url, headers=headers, data=data)
        resp = resp.json()["choices"][0]["message"]
        
        messages.append(resp)
        
        # 检查是否有工具调用
        if not(resp.get("tool_calls")):
            # 模型不再调工具，返回最终回答
            print("模型直接回答，不再调工具")
            return resp["content"]
        
        # 处理本轮所有的工具调用：受到当前模型限制，不支持并行工具调用（Parallel Tool Calls）
        for tc in resp["tool_calls"]:
            func_name = tc["function"]["name"]
            func_args = json.loads(tc["function"]["arguments"])
            print(f"模型决定调用：{func_name}({func_args})")
            
            # 执行函数
            if func_name in available_functions:
                try:
                    func_to_call = available_functions[func_name]
                    function_result = func_to_call(**func_args)
                    print(f"函数返回：{function_result[:50]}...")
                except Exception as e:
                    function_result = f"函数执行出错：{str(e)}"
                    print(f"函数执行出错：{e}")
            else:
                function_result = f"错误：未找到函数 {func_name}"
                print(f"错误：未找到函数 {func_name}")
            
            # 把结果送回给模型
            messages.append({
                "role":"tool",
                "tool_call_id":tc["id"],
                "content": function_result
            })
            
    # 超过最大轮数，强制让模型总结
    messages.append({
        "role":"user",
        "content":"请基于已有的信息，给出最终消息"
    })
    
    # 调模型
    data = json.dumps({
            "model": "nvidia/nemotron-3.5-lightning:free",
            "messages":messages,
        })
    resp = requests.post(url=url, headers=headers, data=data)
    return resp.json()["choices"][0]["message"]["content"]


### 测试 1：需要连续调两个工具的问题

In [7]:
result = run_agent_multi_turn("BOM-1001 变更后，受影响的物料库存够吗？", tools_v2, available_functions_v2)
print("\n=== 最终回答 ===")
print(result)


--- 第 1 轮调用 ---
模型决定调用：get_bom_change({'bom_id': 'BOM-1001'})
函数返回：影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2...

--- 第 2 轮调用 ---
模型决定调用：get_material_stock({'material_name': '螺丝 M3'})
函数返回：库存充足，现有 5000 个...

--- 第 3 轮调用 ---
模型决定调用：get_material_stock({'material_name': '钣金件 A12'})
函数返回：库存紧张，仅剩 120 个，建议补货...

--- 第 4 轮调用 ---
模型直接回答，不再调工具

=== 最终回答 ===
BOM‑1001 变更后受影响的两种物料库存情况如下：

| 物料 | 当前库存 | 评价 |
|------|----------|------|
| 螺丝 M3 | 5000 个 | **库存充足** |
| 钣金件 A12 | 120 个 | **库存不足**，建议补货 |

**结论**：螺丝 M3 的库存足以满足变更需求，而钣金件 A12 的现有库存已不足，需补货后才可保证生产顺利进行。


### 测试 2：不需要调工具的简单问题

In [8]:
result = run_agent_multi_turn(
    "你好，今天有什么可以帮助你的？",
    tools_v2,
    available_functions_v2
)
print("\n=== 最终回答 ===")
print(result)


--- 第 1 轮调用 ---
模型直接回答，不再调工具

=== 最终回答 ===
您好！我是制造企业的 AI 助理，可以帮助您查询 BOM 变更影响、物料库存等信息。请问有什么具体任务需要我协助？


## 第 3 步：加错误处理和降级策略
真实环境中，API 可能超时、数据库可能连不上、参数可能不合法。我们需要让 Agent 优雅处理这些情况。

In [10]:
# 先故意制造一个会出错的工具：

def get_bom_detail(bom_id):
    """
    查询 BOM 详细信息（模拟偶尔会出错）
    """
    import random
    
    # 模拟 30% 概率出错
    if random.random() < 0.3:
        raise ConnectionError("数据库连接超时，请稍后重试")
    
    detail_data = {
        "BOM-1001": {"status": "已发布", "version": "v2.1", "creator": "张三"},
        "BOM-1002": {"status": "审核中", "version": "v1.0", "creator": "李四"},
    }
    return json.dumps(detail_data.get(bom_id, {}), ensure_ascii=False)

In [11]:
# 把这个工具加入现有工具列表
tools_v3 = tools_v2 + [
    {
        "type": "function",
        "function": {
            "name": "get_bom_detail",
            "description": "查询 BOM 的详细信息，包括状态、版本、创建人",
            "parameters": {
                "type": "object",
                "properties": {
                    "bom_id": {
                        "type": "string",
                        "description": "BOM 编号，如 BOM-1001"
                    }
                },
                "required": ["bom_id"]
            }
        }
    }
]

available_functions_v3 = {
    **available_functions_v2,
    "get_bom_detail": get_bom_detail
}

In [12]:
# 多跑几次，看能不能触发错误
for i in range(5):
    print(f"\n===== 第 {i+1} 次测试 =====")
    result = run_agent_multi_turn(
        "BOM-1001 的详细信息是什么？",
        tools_v3,
        available_functions_v3
    )
    print("最终回答：", result)


===== 第 1 次测试 =====

--- 第 1 轮调用 ---
模型决定调用：get_bom_detail({'bom_id': 'BOM-1001'})
函数返回：{"status": "已发布", "version": "v2.1", "creator": "张...

--- 第 2 轮调用 ---
模型直接回答，不再调工具
最终回答： BOM‑1001 的详细信息如下：

- **状态**：已发布  
- **版本**：v2.1  
- **创建人**：张三  

如需进一步信息（如 BOM 物料清单或变更影响），请告知。

===== 第 2 次测试 =====

--- 第 1 轮调用 ---
模型决定调用：get_bom_detail({'bom_id': 'BOM-1001'})
函数返回：{"status": "已发布", "version": "v2.1", "creator": "张...

--- 第 2 轮调用 ---
模型直接回答，不再调工具
最终回答： BOM‑1001 的详细信息如下：

- **状态**：已发布  
- **版本**：v2.1  
- **创建人**：张三

===== 第 3 次测试 =====

--- 第 1 轮调用 ---
模型决定调用：get_bom_detail({'bom_id': 'BOM-1001'})
函数返回：{"status": "已发布", "version": "v2.1", "creator": "张...

--- 第 2 轮调用 ---
模型直接回答，不再调工具
最终回答： BOM-1001 的详细信息如下：

- **状态**：已发布  
- **版本**：v2.1  
- **创建人**：张三

===== 第 4 次测试 =====

--- 第 1 轮调用 ---
模型决定调用：get_bom_detail({'bom_id': 'BOM-1001'})
函数返回：{"status": "已发布", "version": "v2.1", "creator": "张...

--- 第 2 轮调用 ---
模型直接回答，不再调工具
最终回答： BOM‑1001 当前状态：**已发布**，版本 **v2.1**，由 **张三** 创建。

===== 第 5 次测试 

正常时：返回 BOM 的状态、版本、创建人  
出错时：模型会说“查询暂时不可用，请稍后重试”而不是直接崩溃

## 第 4 步：带记忆的 Agent——记住对话上下文
目前的 Agent 每次调用都是独立的，不记得上一句说了什么。今天最后一步，我们要让 Agent 能维持多轮对话。  
实现了 ConversationalAgent 类，支持多轮对话记忆

In [16]:
#     当你执行 obj = MyClass(arg) 时，Python 底层实际上执行了两个步骤：
#     1.调用 __new__ 创建实例。
#     2.调用 __init__(instance, arg) 进行初始化。
#     解释器会‌自动‌将新创建的实例作为第一个参数传递给 __init__。因此，你的方法定义中必须有一个位置来接收这个“自动传入”的实例。
class Person:
    def __init__(self, name):
        self.name = name  # self 指向当前实例

p = Person("Alice")
print(p.name)  # 输出: Alice

Alice


In [14]:
import requests
import json

class ConversationalAgent:
    """
    带记忆的对话式 Agent
    """

    def __init__(self, tools, available_functions, api_key, system_prompt=None):
        self.tools = tools
        self.available_functions = available_functions
        
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.api_key = api_key
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-type": "application/json"
        }
        self.system_prompt = system_prompt or "你是制造企业的 AI 助理，回答简洁专业，用中文。"
        self.messages = [
            {"role":"system", "content": self.system_prompt}
        ]
    
    def chat(self, user_input, max_turns=5):
        """发送消息并返回回答，自动维护对话历史"""
        
        self.messages.append({
            "role":"user","content":user_input
        })
        
        turn_count = 0
        
        while turn_count < max_turns:
            
            turn_count += 1
        
            data = json.dumps({
                "model":"nvidia/nemotron-3.5-lightning:free",
                "messages": self.messages,
                "tools":self.tools,
                "tool_choice":"auto"
            })
            resp = requests.post(url=self.url, headers=self.headers, data=data)
            resp = resp.json()["choices"][0]["message"]
            
            self.messages.append(resp)
            
            if not (resp.get("tool_calls")):
                return resp["content"]
            
            for tc in resp["tool_calls"]:
                func_name = tc["function"]["name"]
                func_args = json.loads(tc["function"]["arguments"])
                
                
                if func_name in self.available_functions:
                    func_to_call = self.available_functions[func_name]
                    funciton_result = func_to_call(**func_args)
                    
                    self.messages.append({
                        "role":"tool",
                        "tool_call_id":tc["id"],
                        "content":funciton_result
                    })
        # 超过最大轮数
        self.message.append({
            "role":"user",
            "content":"请基于已有信息给出最终回答。"
        })
        
        data = json.dumps({
            "model":"nvidia/nemotron-3.5-lightning:free",
            "messages":self.message
        })
        
        resp2 = requests.post(url=url, headers=headers, data=data)
        return resp2.json()["choices"][0]["message"]["content"]
                
    def reset(self):
        """重置对话历史"""
        seld.message = [
            {"role":"system", "content":self.system_prompt}
        ]

### 测试多轮对话

In [24]:
# 创建 Agent 实例

agent = ConversationalAgent(tools_v2, available_functions_v2, api_key)

# 第一轮对话
print("用户：BOM-1001 变更影响什么？")
result1 = agent.chat("BOM-1001 变更影响什么？")
print(f"助手：{result1}\n")

# 第二轮对话（依赖第一轮的上下文）
print("用户：那它的库存够吗？")
result2 = agent.chat("那它的库存够吗？")
print(f"助手：{result2}\n")

# 第三轮对话（换个话题）
print("用户：BOM-1002 呢？")
result3 = agent.chat("BOM-1002 呢？")
print(f"助手：{result3}\n")

# 第四轮对话（跨话题追问）
print("用户：刚才说的 BOM-1001 的版本是多少？")
result4 = agent.chat("刚才说的 BOM-1001 的版本是多少？")
print(f"助手：{result4}")

用户：BOM-1001 变更影响什么？
助手：BOM‑1001 的变更影响以下物料和产线：

- **受影响物料**：螺丝 M3、钣金件 A12  
- **涉及产线**：SMT‑2  

如需进一步库存或生产计划信息，请告知。

用户：那它的库存够吗？
助手：**库存情况（ BOM‑1001 变更）**  

| 物料 | 当前库存 | 备注 |
|------|----------|------|
| **螺丝 M3** | 5,000 个 | 足量，满已满 |
| ** A12** | 120 个 | 余量较较已紧较 = 120，建议根据产计划补补充 |

如需需库存不足的 ** A12**，请尽快补补补充或调调重新评BOM。如果需要需要BOM的其他信息或后续操检索请告诉我告知。

用户：BOM-1002 呢？
助手：BOM‑1002 的变更影响以下物料和产线：

- **受影响物料**：电池 5000 mAh、外壳 B 款  
- **涉及产线**：组装‑3  

如需查询这两种物料的库存情况，请告诉我。

用户：刚才说的 BOM-1001 的版本是多少？
助手：目前系统返回的 BOM‑1001 变更信息中未包含版本号，仅列出受影响的物料和产线。如需确认具体 BOM 版本，请联系工程部或提供其他检索条件，我可以进一步协助查询。


## 第 5 步：整合——做一个完整的 BOM 变更通知 Agent

In [17]:
def send_feishu_message(message):
    """
    模拟发送飞书消息（实际项目中会调飞书 API）
    """
    print(f"[飞书消息已发送]: {message}")
    return f"[飞书消息已发送]: {message[:30]}..."

# 完整工具集
tools_complete = [
    {
        "type": "function",
        "function": {
            "name": "get_bom_change",
            "description": "根据 BOM 编号查询变更影响到的物料和产线",
            "parameters": {
                "type": "object",
                "properties": {
                    "bom_id": {
                        "type": "string",
                        "description": "BOM 编号，如 BOM-1001"
                    }
                },
                "required": ["bom_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_material_stock",
            "description": "查询某个物料的当前库存情况",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_name": {
                        "type": "string",
                        "description": "物料名称，如 螺丝 M3"
                    }
                },
                "required": ["material_name"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"send_feishu_message",
            "description":"发送飞书消息通知相关人员，当需要通知产线负责人或物料管理员时调用",
            "parameters":{
                "type":"object",
                "properties":{
                    "message":{
                        "type":"string",
                        "description":"要发送的消息内容"
                    }
                },
                "required":["message"]
            }
        }
    }
]

available_functions_complete = {
    "get_bom_change": get_bom_change,
    "get_material_stock": get_material_stock,
    "send_feishu_message":send_feishu_message
}

### 测试完整的 BOM 变更通知流程：

In [18]:
system_prompt = """你是制造企业的 AI 助理。你的工作流程：
1. 当用户查询 BOM 变更影响时，先查变更影响
2. 然后查受影响物料的库存
3. 如果库存不足，发送飞书通知提醒补货
4. 最后给出完整的分析和建议
"""
agent_complete = ConversationalAgent(tools_complete, available_functions_complete, api_key, system_prompt)
result = agent_complete.chat("BOM-1001 发生了变更，请评估影响并通知相关人员。")
print("\n=== 最终回答 ===")
print(result)

[飞书消息已发送]: BOM-1001 变更影响物料分析：
- 螺丝 M3：库存充足，现有 5000 个
- 钣金件 A12：库存紧张，仅剩 120 个，请及时补货

请相关物料管理员注意补货，避免生产中断。

=== 最终回答 ===
**BOM‑1001 变更影响评估报告**

| 受影响物料 | 当前库存 | 状态 | 建议 |
|-----------|----------|------|------|
| 螺丝 M3   | 5000 个  | **库存充足** | 无需立即操作，继续正常生产 |
| 钣金件 A12| 120 个   | **库存紧张** | **请及时补货**，以免影响产线 SMT-2 的生产计划 |

### 工作流程执行情况
1. **查询 BOM 变更影响** – 通过 `get_bom_change` 获取受影响物料（螺丝 M3、钣金件 A12）和涉及产线（SMT-2）。  
2. **检查库存情况** –  
   - `get_material_stock` 显示螺丝 M3 有 5000 个，库存充足。  
   - `get_material_stock` 显示钣金件 A12 仅剩 120 个，库存不足。  
3. **发送飞书通知** – 已调用 `send_feishu_message`，通知相关物料管理员和产线负责人库存状态并请求补货。  
4. **给出分析与建议** – 见下文。

### 发送的飞书消息
```
BOM-1001 变更影响物料分析：
- 螺丝 M3：库存充足，现有 5000 个
- 钣金件 A12：库存紧张，仅剩 120 个，请及时补货

请相关物料管理员注意补货，避免生产中断。
```

### 推荐后续动作
1. **补货钣金件 A12**  
   - 采购或仓库部门应在收到通知后 24 小时内启动补货流程，确保至少恢复至生产计划所需的安全库存水平（建议至少 500 件，视 SMT-2 周产量而定）。  
2. **监控螺丝 M3**  
   虽然当前充足，建议每日检查库存变化，特别是在 BOM-1001 的大批量生产周期内，防止意外短缺。  
3. **更新 BOM 变更记录**  
   将本次变更及库存评估结果同步至企业的 BOM 管理系统，以便后续变更评估参考。  
4. **生

 ​ Agent 自动完成：查变更 → 查库存 → 判断是否需要通知 → 发送通知 → 给出总结